In [1]:
import pandas as pd
raw_vitals_file_path = '/home/guido/Code/charite/baselines/data/processed/raw_vitals.parquet'
raw_labs_file_path = '/home/guido/Code/charite/baselines/data/processed/raw_labs.parquet'
raw_vitals = pd.read_parquet(raw_vitals_file_path)
raw_labs = pd.read_parquet(raw_labs_file_path)

In [2]:
raw_vitals

,stay_id,subject_id,hours_since_icu_in,feature_name,value
0,37081114,10000690,0.283333,heart_rate,79.0
1,37081114,10000690,0.283333,sbp,107.0
2,37081114,10000690,0.283333,dbp,63.0
3,37081114,10000690,0.283333,respiratory_rate,23.0
4,37081114,10000690,0.283333,spo2,100.0
...,...,...,...,...,...
206461,34921121,10116099,41.850000,respiratory_rate,17.0
206462,34921121,10116099,41.850000,spo2,95.0
206463,34921121,10116099,42.850000,heart_rate,50.0
206464,34921121,10116099,42.850000,respiratory_rate,18.0


In [4]:

raw_vitals.nunique()

stay_id                 963
subject_id              710
hours_since_icu_in    37506
feature_name              6
value                   340
dtype: int64

In [5]:
raw_labs

,stay_id,subject_id,hours_since_icu_in,feature_name,value
0,37081114,10000690,7.316667,hematocrit,28.5
1,37081114,10000690,7.316667,hemoglobin,9.5
2,37081114,10000690,7.316667,platelets,199.0
3,37081114,10000690,7.316667,wbc,7.5
4,37081114,10000690,7.316667,chloride,104.0
...,...,...,...,...,...
36121,34921121,10116099,47.766667,creatinine,2.9
36122,34921121,10116099,47.766667,glucose,152.0
36123,34921121,10116099,47.766667,potassium,4.8
36124,34921121,10116099,47.766667,sodium,131.0


In [6]:
raw_labs.nunique()

stay_id                956
subject_id             704
hours_since_icu_in    5808
feature_name            11
value                  966
dtype: int64

In [ ]:
import numpy as np

test = np.load('/home/guido/Code/charite/baselines/data/processed/test.npz')



AttributeError: 'NpzFile' object has no attribute 'head'

In [12]:
test.keys()

KeysView(NpzFile '/home/guido/Code/charite/baselines/data/processed/test.npz' with keys: data, orig_mask, stay_ids, subject_ids)

In [7]:
import timesfm
import inspect
# Questo stamperà esattamente cosa si aspetta la tua versione di TimesFm
print(f"Argomenti accettati: {inspect.signature(timesfm.TimesFm.__init__)}")

Argomenti accettati: (self, hparams: timesfm.timesfm_base.TimesFmHparams, checkpoint: timesfm.timesfm_base.TimesFmCheckpoint) -> None


In [ ]:
import torch
import os
import timesfm
import numpy as np
from safetensors.torch import load_file

# 1. Configurazione Percorsi
model_dir = os.path.expanduser("~/timesfm_fix/model_weights/")
safetensors_path = os.path.join(model_dir, "model.safetensors")
ckpt_path = os.path.join(model_dir, "torch_model.ckpt")

# 2. Caricamento del dizionario dei pesi (State Dict)
if os.path.exists(ckpt_path):
    print("🧠 Caricamento pesi da torch_model.ckpt...")
    old_sd = torch.load(ckpt_path, map_location="cpu")
elif os.path.exists(safetensors_path):
    print("🧠 Caricamento pesi da model.safetensors...")
    old_sd = load_file(safetensors_path)
else:
    raise FileNotFoundError(f"❌ Non ho trovato né .ckpt né .safetensors in {model_dir}")

# 3. Mappa di traduzione (Google Legacy -> New Standards)
new_sd = {}
mapping = {
    "tokenizer": "input_ff_layer",
    "stacked_xf": "stacked_transformer.layers",
    "output_projection_point": "horizon_ff_layer",
    "output_projection_quantiles": "horizon_ff_layer_quantiles",
    "attn.qkv_proj": "self_attn.qkv_proj",
    "attn.out": "self_attn.o_proj",
    "ff0": "mlp.gate_proj",
    "ff1": "mlp.down_proj",
    "pre_attn_ln": "input_layernorm",
    "pre_ff_ln": "mlp.layer_norm",
}

for k, v in old_sd.items():
    new_k = k
    for old_name, new_name in mapping.items():
        new_k = new_k.replace(old_name, new_name)
    
    # Correzione specifica per gli strati 'hidden' (richiedono .0.)
    if "hidden_layer" in new_k and ".0." not in new_k:
        new_k = new_k.replace("hidden_layer", "hidden_layer.0")
    new_sd[new_k] = v

# 4. Inizializzazione modello
hparams = timesfm.TimesFmHparams(
    context_len=1024,
    horizon_len=12,
    input_patch_len=32,
    output_patch_len=128,
    num_layers=20,
    model_dims=1280,
    backend="torch",
)

model = timesfm.TimesFm(hparams=hparams, checkpoint=None)

# 5. Iniezione pesi e test
try:
    model._model.load_state_dict(new_sd, strict=False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model._model.to(device)
    print("🚀 VITTORIA! Modello caricato e mappato.")
    
    # Test veloce
    data = np.random.randn(1, 1024).astype(np.float32)
    forecast, _, _ = model.forecast(data, freq=[0])
    print(f"✨ Previsione completata: {forecast[0][:3]}")
except Exception as e:
    print(f"❌ Errore: {e}")

FileNotFoundError: ❌ Non ho trovato né .ckpt né .safetensors in /home/guido/timesfm_fix/